# SubSentry: Customer Churn Prediction Engine
### HackAttack 2026 - AI Customer Success Platform

This notebook trains the predictive backend classifier for **SubSentry** using the **IBM Telco Customer Churn dataset**. 
We train a Random Forest model to analyze contract details, monthly charges, and services to output churn probabilities.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

### 1. Load the IBM Dataset

In [ ]:
df = pd.read_csv(r'C:\Users\shenyee\Desktop\HackAttack 2026\ai-customer-success\pulse360\backend\plots\Telco-Customer-Churn.csv')
df.head()

### 2. Preprocessing & Feature Engineering
- Convert `TotalCharges` to numeric.
- Drop rows with missing values.
- Map binary target `Churn` to 0 and 1.
- Encode categorical variables using one-hot dummies.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')
df = df.dropna()
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

df_clean = df.drop(columns=['customerID'])
X = pd.get_dummies(df_clean.drop(columns=['Churn']), drop_first=True)
y = df_clean['Churn']

print(f'Features shape: {X.shape}')

### 3. Split & Train Random Forest Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

### 4. Evaluate and Generate Pitch Plots

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('Classification Report:')
print(classification_report(y_test, y_pred))

#### Plot Confusion Matrix

In [ ]:
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', cbar=False,
            xticklabels=['Retained', 'Churned'], yticklabels=['Retained', 'Churned'],
            annot_kws={'size': 14, 'weight': 'bold'})
plt.title('SubSentry Model: Confusion Matrix', fontsize=14, weight='bold', pad=15)
plt.ylabel('Actual Label', fontsize=12, labelpad=10)
plt.xlabel('Predicted Label', fontsize=12, labelpad=10)
plt.tight_layout()
plt.show()

#### Plot ROC-AUC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5.5))
plt.plot(fpr, tpr, color='#9D6638', lw=3, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='#4E220F', lw=1.5, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=11, labelpad=8)
plt.ylabel('True Positive Rate', fontsize=11, labelpad=8)
plt.title('SubSentry Churn Engine: Receiver Operating Characteristic (ROC)', fontsize=13, weight='bold', pad=15)
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.15)
plt.tight_layout()
plt.show()